# Car Price Prediction


#### Project Workflow
1. Understand the Dataset
- Review all columns and their meanings (you’ve already done this — great start!)
- Identify which variables are:
- Independent (features)
- Dependent (target)

2. Clean and Prepare the Data
- Check for missing values or anomalies (e.g., nulls sales)
- Create new features if needed:

3. Explore the Data (EDA)
Use visualizations to uncover patterns:
- 📉 Boxplots to see sales distribution by weather or promotion
- 📌 Correlation heatmap to see which features influence sales most

4. Model Sales Drivers
- Use regression models (e.g., Linear Regression, Random Forest, XGBoost) to predict daily_sales
- Evaluate feature importance: which variables drive sales the most?
- Try time series models (e.g., ARIMA, Prophet) if you're forecasting future sales

7. Present Your Work
- Create a dashboard (Excel, Power BI, or Tableau)
- Summarize key insights in a slide deck or report
- Include visuals, trends, and actionable takeaways


In [ ]:
# Importing Libraries

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns


In [ ]:
#Importing Dataset
df = pd.read_csv('car_sales_data.csv')

raw_df = df.copy()

## Data Cleaning

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.duplicated().sum()

In [ ]:
df[df.duplicated()]

In [ ]:
# Handling Duplicates
df = df.drop_duplicates()

In [ ]:
df.duplicated().sum()

In [ ]:
df = df.reset_index(drop=True)

In [ ]:
# Handling Missing Values
df.isnull().sum()

## EDA


✅ 1. Univariate Analysis (Quick Checks)
✔ Histograms / KDE plots for:
- Price
- Mileage
- Year of manufacture
- Engine size


✅ 2. Outlier Detection (Outliers can ruin models if not handled.)
Use: Boxplots, IQR, Scatterplots of Price vs key features
Look specifically for:
- Extremely old cars
- Very high mileage cars
- Price values too low or too high compared to the rest


✅ 3. Relationship Analysis (Minimal Pairwise Checks)
Only 3 scatterplots are needed:
- Price vs Mileage (declining trend?)
- Price vs Age/Year (strong expected relationship)
- Price vs Engine size (positive relationship)
Purpose:
Helps you see whether linear or non-linear models might work better, and where transformations may help.


✅ 4. Categorical Variable Insights
✔ View counts of: Manufacturer, Model, Fuel type
`df['Manufacturer'].value_counts()`
`df['Model'].value_counts()`
`df['Fuel type'].value_counts()`

Why this matters:
- Helps detect rare categories
- Helps decide which encoding to use
- Model will likely need target/frequency encoding
- Manufacturer/Fuel type → one-hot is fine


✅ 5. Multicollinearity Check (Optional but Recommended)

Mainly to avoid redundant variables:

- Year vs Car age (if you eventually create it)
- Engine size vs Engine category
- Mileage vs Mileage per year (after engineering)

For now, just check correlation among numerical columns


`NB: Do No7 after Completing Data Preprocessing and Feature Engineering`

### Univariate Analysis; To detect outliers and Skeweness
✔ Histograms / KDE and boxplots for:
- Price
- Mileage
- Year of manufacture
- Engine size

In [ ]:
# Visualizing Histogram + KDE for numerical columns
numerical_features = ['Engine size', 'Year of manufacture', 'Mileage', 'Price']

plt.figure(figsize=(12, 8))

for i, feature in enumerate(numerical_features, 1):
    plt.subplot(2, 2, i)
    sns.histplot(df[feature], kde=True)
    plt.title(f'Distribution of {feature}')

plt.tight_layout()
plt.show()

In [ ]:
# Visualizing boxplots for numerical columns
numerical_features = ['Engine size', 'Year of manufacture', 'Mileage', 'Price']

plt.figure(figsize=(12, 8))

for i, feature in enumerate(numerical_features, 1):
    plt.subplot(2, 2, i)
    sns.boxplot(x=df[feature])
    plt.title(f'Boxplot of {feature}')

plt.tight_layout()
plt.show()


In [ ]:
df[numerical_features].describe()

### Looking for Correlations

In [ ]:
df.dtypes

In [ ]:
corr_matrix = df[numerical_features].corr()

In [ ]:
# Visualizing the correlation matrix using a heatmap

plt.figure(figsize=(10, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix of Numeric Features')
plt.show()

From the correlation matrix above;
- Year of manufacture has a strong direct proportionality to Price
- Engine size is moderately directly proportional to price
- Mileage is Strongly Indirectly Proportional to the Price

In [ ]:
df.head(10)

### Relationship / Multivariate Analysis


In [ ]:
#Price vs Mileage Scatter Plot
sns.scatterplot(data=df, x="Mileage", y="Price", alpha=0.4)
plt.title('Price vs Mileage')
plt.show()

`Theres a downward trend i.e; the lower the milage the higher the price.`

In [ ]:
#Price vs Age Scatter Plot
#First create a new column 'Age' because Year of manufacture is not very intuitive
df['Age'] = 2022 - df['Year of manufacture'] #Assuming the current year is 2022


sns.scatterplot(data=df, x="Age", y="Price", alpha=0.4)
plt.title('Price vs Age')
plt.show()

`Theres a downward trend as well in the price vs Age i.e; the lower the older the age, lower the price.`

In [ ]:
# Boxplot for Price vs Engine size
sns.boxplot(data=df, x="Engine size", y="Price")
plt.title("Price vs Engine size(Boxplot)")
plt.show()

In [ ]:
# Price vs Manufacturer Boxplot
plt.figure(figsize=(10,6))
sns.boxplot(data=df, x="Manufacturer", y="Price")
plt.xticks(rotation=45)
plt.title("Price vs Manufacturer")
plt.show()

`In order of price: Porsche > BMW > Toyota > Ford > VW`

In [ ]:
#Price vs Fuel type
sns.boxplot(data=df, x="Fuel type", y="Price")
plt.title("Price vs Fuel type")
plt.show()

`In order of price: Hybrid > Diesel > Petrol`

### Categorical Variable Insights
✔ View counts of: Manufacturer, Model, Fuel type
`df['Manufacturer'].value_counts()`
`df['Model'].value_counts()`
`df['Fuel type'].value_counts()`

Why this matters:
- Helps detect rare categories
- Helps decide which encoding to use
- Model will likely need target/frequency encoding
- Manufacturer/Fuel type → one-hot is fine

In [ ]:
df['Manufacturer'].value_counts()


In [ ]:
print(df['Model'].value_counts())

In [ ]:
df['Model'].unique()

In [ ]:

df['Fuel type'].value_counts()

`NB: When performing encoding on the categorical colums in the Preprocessing steps;`

Manufacturer and Fuel type --> One-hot encoding 

Model --> Target encoding

`Also, Perform the Target encoding on the training set alone rather than on the whole daset so as to avoid feature leakage`

# Data Preprocessing

✅ 1. Essential Feature Engineering for Car Price Prediction

- Car Age (instead of using Year directly)
`df['Car_Age'] = current_year - df['Year of manufacture']`

- Mileage per Year (Usage Intensity). i.e A car with 150,000 km over 15 years is not the same as 150,000 km over 5 years.
`df['Mileage_per_Year'] = df['Mileage'] / df['Car_Age'].replace(0, 1)`

- Engine Size Category (optional binning)
You can create bins like:
Small: <1.4
Medium: 1.4–2.0
Large: >2.0
Or just standard binning:
`df['Engine_Category'] = pd.cut(df['Engine size'], bins=[0, 1.4, 2.0, 4.0], labels=['Small','Medium','Large'])`

- Log-transform the target (Price): Car prices are usually right-skewed. Applying log transformation stabilizes variance and reduces outlier impact
`df['Log_Price'] = np.log(df['Price'])`


✅ 2. Encoding Categorical Variables Properly
Thefore the ffg encoding is prefarable;
Manufacturer → One-hot encoding is fine.
Fuel Type → One-hot encoding works.
Model → This is tricky.


✅ 3. Interaction features (often useful)
Useful examples:
Engine size × Manufacturer
Engine size × Fuel type
Car age × Mileage

✅ 4. Remove or treat extreme outliers.
Before fitting the model:
-Remove cars priced at absurdly low or high values
- Remove mileage above the 99th percentile
- Remove cars older than 30 years if they skew the model

eg: Your dataset has a 1988 Toyota — that’s essentially a classic/vintage category and will distort the model.

## Feature Engineering
- Age
- Mileage per year
- Brand popularity

In [ ]:
# We've donr the feature engineering for 'Age' column already in the EDA section
print(df['Age'])

In [ ]:
df['Mileage']

In [ ]:
df['Mileage_per_year'] = df['Mileage'] / df['Age'].replace(0,1)
print(df['Mileage_per_year'])

In [ ]:
brand_freq = df['Manufacturer'].value_counts(normalize=True)
df['Brand_popularity'] = df['Manufacturer'].map(brand_freq)


In [ ]:
df['Brand_popularity']

In [ ]:
# Handling Missing Values
df.isna().sum()


## Encoding Categorical Variables
- Manufacturer
- Fuel type
- Model (Target encoding to be done after splitting the dataset)

In [ ]:
print(df['Manufacturer'].nunique())

print(df['Manufacturer'].unique())

In [ ]:
print(df['Model'].nunique())

print(df['Model'].unique())

In [ ]:
print(df['Fuel type'].nunique())

print(df['Fuel type'].unique())

In [ ]:
df.columns

In [ ]:
# One-hot encoding for Manufacturer and Fuel type

cols_to_encode = ['Manufacturer', 'Fuel type']

# Creating Dummy Variables for manufacturer and fuel type
df = pd.get_dummies(df, columns=cols_to_encode, drop_first=True)


In [ ]:
df.columns

In [ ]:
df

### Transformation of numerical features
🔵 If the feature is moderately right-skewed → use Log Transformation
Use log when:

Values are positive only (no zeros, no negatives)

The tail is long but not extreme

Examples in your dataset:
✔ Price
✔ Mileage


🟢 If the feature is highly right-skewed → use Square Root or Cube Root

Use this when log is still not enough OR when there are many small values.

Works well for:
✔ Engine Size (mild skew but discrete peaks)
✔ Mileage (if log still leaves skew)


`NB After applying transformation, replot the histogram + KDE and boxplot to check if the features are still skewed. Also, remember that XGBoost model does not require transformation` 

In [ ]:
# Log Transformation of the mildely skewed numerical features
df['Price_log'] = np.log1p(df['Price'])
df['Mileage_log'] = np.log1p(df['Mileage'])
df['Mileage_per_year_log'] = np.log1p(df['Mileage_per_year'])


#--------------------------
#sqrt or cube root transformation 
df['Mileage_sqrt'] = np.sqrt(df['Mileage'])


In [ ]:
# Plotting log and sqrt transformations of Mileage for comparison
plt.figure(figsize=(12, 5))

# Log transformation
plt.subplot(1, 2, 1)
sns.histplot(np.log1p(df['Mileage']), kde=True)
plt.title('Log1p Transformation of Mileage')

# Square root transformation
plt.subplot(1, 2, 2)
sns.histplot(np.sqrt(df['Mileage']), kde=True)
plt.title('Square Root Transformation of Mileage')

plt.tight_layout()
plt.show()

In [ ]:
df.columns

In [ ]:
# Visualizing Histogram + KDE for numerical columns
transformed_features = ['Mileage_log', 'Mileage_sqrt']

plt.figure(figsize=(12, 8))

for i, feature in enumerate(transformed_features, 1):
    plt.subplot(2, 2, i)
    sns.histplot(df[feature], kde=True)
    plt.title(f'Distribution of {feature}')

plt.tight_layout()
plt.show()

Based on the plots:

- `Mileage_log:` The log-transformed distribution is much closer to normal (bell-shaped), with reduced skewness and a more symmetric appearance.
- `Mileage_sqrt:` The square root transformation also improves the distribution, but it is still more peaked and less symmetric than the log transformation.
`NB:`
The log transformation (Mileage_log) is the better choice for your Mileage variable. It produces a distribution that is more suitable for most regression and statistical modeling tasks, especially when using linear models.

Therefore, Keep and use the Mileage_log feature for your modeling. You can drop or ignore the Mileage_sqrt column.

# Train / Test Split

We spliting the dataset into 3 (training set, test(or validation) set and app(To be used on the streamlit app))

In [ ]:
#Separating Features and Target Variable

X = df.drop(columns=['Price_log', 'Price']) #Features
y = df['Price_log'] #Target

In [ ]:
# 1st split: Hold-out set (never touched until the end)

from sklearn.model_selection import train_test_split

# First split: Hold-out set (never touched until the end)
X_temp, X_app, y_temp, y_app = train_test_split(
    X, y, test_size=0.15, random_state=42
)


In [ ]:
# 2nd split: Training and test sets

X_train, X_test, y_train, y_test = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42
)



### Multicollinearity Check (To be done after the Preprocessing step)

Mainly to avoid redundant variables:

- Year vs Car age (if you eventually create it)
- Engine size vs Engine category
- Mileage vs Mileage per year (after engineering)

For now, just check correlation among numerical columns

In [ ]:
num_features = X_train.select_dtypes(include='number')


In [ ]:
print(num_features.columns)

In [ ]:
# Visualizing the correlation matrix using a heatmap

plt.figure(figsize=(10, 6))
sns.heatmap(num_features.corr(), annot=True, cmap='coolwarm')
plt.show()


In [ ]:
print(X_train.dtypes)

In [ ]:
# ===== Branch for pipeline-based training =====
X_train_pipe = X_train.copy()
X_train_pipe = X_train_pipe.drop(columns=['Year of manufacture', 'Mileage_log', 'Mileage_sqrt', 'Mileage_per_year', 'Mileage_per_year_log'])
y_train_pipe = y_train.copy()


X_test_pipe = X_test.copy()
X_test_pipe = X_test_pipe.drop(columns=['Year of manufacture', 'Mileage_log', 'Mileage_sqrt', 'Mileage_per_year', 'Mileage_per_year_log'])
y_test_pipe = y_test.copy()


In [ ]:
# Droping Redundant Columns

X_train = X_train.drop(columns=['Year of manufacture', 'Mileage', 'Mileage_sqrt', 'Mileage_per_year', 'Mileage_per_year_log'])


In [ ]:
# apply same drop to test/validation set and application set

X_test = X_test.drop(columns=['Year of manufacture', 'Mileage', 'Mileage_sqrt', 'Mileage_per_year', 'Mileage_per_year_log'])

# Drop Mileage_log cos it won't be needed for the input during deployment 
X_app = X_app.drop(columns=['Year of manufacture', 'Mileage_log', 'Mileage_sqrt', 'Mileage_per_year', 'Mileage_per_year_log'])

In [ ]:
# Saving the application test set to CSV files
app_test_features_df = X_app.copy()
app_test_target_df = y_app.copy()


app_test_features_df.to_csv('app_test_features.csv', index=False)
app_test_target_df.to_csv('app_test_target.csv', index=False)


### Variance Inflation Factor (VIF) — the real check

In [ ]:
num_features = X_train.select_dtypes(include='number')


In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_data = pd.DataFrame()
vif_data["feature"] = num_features.columns
vif_data["VIF"] = [
    variance_inflation_factor(num_features.values, i)
    for i in range(num_features.shape[1])
]

vif_data.sort_values(by="VIF", ascending=False)


### Target Encoding for `Model` Column

In [ ]:
import category_encoders as ce

# Initialize the target encoder for the 'Model' column
target_encoder = ce.TargetEncoder(cols=['Model'])


In [ ]:
# Fit on X_train and y_train only (no leakage from X_test or app_test)
target_encoder.fit(X_train['Model'], y_train)


In [ ]:
# Transform X_train
X_train_encoded = X_train.copy()
X_train_encoded['Model'] = target_encoder.transform(X_train['Model'])

# Transform X_test
X_test_encoded = X_test.copy()
X_test_encoded['Model'] = target_encoder.transform(X_test['Model'])


### Feature Scaling

In [ ]:
numeric_features = ['Mileage_log', 'Brand_popularity', 'Engine size', 'Age', 'Model']


In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()


In [ ]:
# Make copies to preserve originals
X_train_scaled = X_train_encoded.copy()
X_test_scaled = X_test_encoded.copy()

# Fit scaler on training data numeric features
X_train_scaled[numeric_features] = scaler.fit_transform(X_train_encoded[numeric_features])

# Transform test set
X_test_scaled[numeric_features] = scaler.transform(X_test_encoded[numeric_features])


# Build the Model
- Linear Regression
- Random Forest Regression
- XGBoost



### Linear Regression Model


In [ ]:
# Fit the Linear Regression Model on the Scaled Data

from sklearn.linear_model import LinearRegression  
regressor = LinearRegression()
regressor.fit(X_train_scaled, y_train)


In [ ]:
# Predicting the Test Set Results
y_pred = regressor.predict(X_test_scaled)

y_pred

In [ ]:
y_test

In [ ]:
# Evaluating model performance

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.2f}")

In [ ]:
# Convert predictions back to actual price
y_test_pred_price = np.expm1(y_pred)
y_test_actual_price = np.expm1(y_test)


In [ ]:
price_mae = mean_absolute_error(y_test_actual_price, y_test_pred_price)
price_rmse = np.sqrt(mean_squared_error(y_test_actual_price, y_test_pred_price))
price_mae, price_rmse,

print(f"MAE: {price_mae:.2f}")
print(f"RMSE: {price_rmse:.2f}")

### Random Forest Regression

In [ ]:
# Fitting the Random Forest Regression Model into the Dataset
from sklearn.ensemble import RandomForestRegressor
rf_regressor = RandomForestRegressor(n_estimators=300, random_state=42)

rf_regressor.fit(X_train_encoded, y_train)


In [ ]:
rf_y_pred = rf_regressor.predict(X_test_encoded)


In [ ]:
rf_y_pred

In [ ]:
y_test.head(10)

In [ ]:
# Evaluating Random Forest model performance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


mae_rf = mean_absolute_error(y_test, rf_y_pred)
rmse_rf = np.sqrt(mean_squared_error(y_test, rf_y_pred))
r2_rf = r2_score(y_test, y_pred)

print(f"Random Forest MAE: {mae_rf:.2f}")
print(f"Random Forest RMSE: {rmse_rf:.2f}")
print(f"Random Forest R² Score: {r2_rf:.2f}")


In [ ]:
# Convert predictions back to actual price
rf_y_test_pred_price = np.expm1(rf_y_pred)
rf_y_test_actual_price = np.expm1(y_test)


In [ ]:
rf_price_mae = mean_absolute_error(rf_y_test_actual_price, rf_y_test_pred_price)
rf_price_rmse = np.sqrt(mean_squared_error(rf_y_test_actual_price, rf_y_test_pred_price))
rf_price_mae, rf_price_rmse,

print(f"Random Forest MAE: {rf_price_mae:.2f}")
print(f"Random Forest RMSE: {rf_price_rmse:.2f}")

In [ ]:
# Feature Importance from Random Forest Model
import pandas as pd

feature_importance = rf_regressor.feature_importances_
features = X_train_encoded.columns

importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': feature_importance
}).sort_values(by='Importance', ascending=False)

In [ ]:
# Visualizing Feature Importance
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.barh(importance_df['Feature'], importance_df['Importance'], color='skyblue')
plt.gca().invert_yaxis()
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

### XGBoost Model

In [ ]:
# Fitting the XGBoost Regression Model into the Dataset using the redudced feature set
from xgboost import XGBRegressor

xgb_regressor = XGBRegressor(n_estimators=300, learning_rate=0.1, max_depth=6, random_state=42)
xgb_regressor.fit(X_train_encoded, y_train)

In [ ]:
xgb_y_pred = xgb_regressor.predict(X_test_encoded)

In [ ]:
xgb_y_pred

In [ ]:
y_test

In [ ]:
# Evaluating XGBoost model pexgbormance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np


mae_xgb = mean_absolute_error(y_test, xgb_y_pred)
rmse_xgb = np.sqrt(mean_squared_error(y_test, xgb_y_pred))
r2_xgb = r2_score(y_test, y_pred)

print(f"XGBoost MAE: {mae_xgb:.2f}")
print(f"XGBoost RMSE: {rmse_xgb:.2f}")
print(f"XGBoost R² Score: {r2_xgb:.2f}")


In [ ]:
# Convert predictions back to actual price
xgb_y_test_pred_price = np.expm1(xgb_y_pred)
xgb_y_test_actual_price = np.expm1(y_test)


In [ ]:
xgb_price_mae = mean_absolute_error(xgb_y_test_actual_price, xgb_y_test_pred_price)
xgb_price_rmse = np.sqrt(mean_squared_error(xgb_y_test_actual_price, xgb_y_test_pred_price))
xgb_price_mae, xgb_price_rmse,

print(f"XGBoost MAE: {xgb_price_mae:.2f}")
print(f"XGBoost RMSE: {xgb_price_rmse:.2f}")

### Preprocessing Pipeline

In [ ]:
X_train.columns

In [ ]:
X_pipeline.columns


In [ ]:
numeric_features = [
    'Engine size',
    'Age',
    'Brand_popularity',
    'Mileage'
]

binary_features = [
    'Manufacturer_Ford',
    'Manufacturer_Porsche',
    'Manufacturer_Toyota',
    'Manufacturer_VW',
    'Fuel type_Hybrid',
    'Fuel type_Petrol'
]

categorical_features = ['Model']


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from category_encoders import TargetEncoder

#---------------------
# Mileage Log Transformer
#---------------------
mileage_log_transformer = FunctionTransformer(
    lambda x: np.log(x + 1),
    feature_names_out="one-to-one"
)


#---------------------
# Numeric Pipeline
#---------------------
numeric_pipeline = Pipeline(steps=[
    ('mileage_log', ColumnTransformer(
        [('log', mileage_log_transformer, ['Mileage'])],
        remainder='passthrough'
    )),
    ('scaler', StandardScaler())
])


#--------------------
# Categorical Pipeline
#--------------------
categorical_pipeline = Pipeline(steps=[
    ('target_encoder', TargetEncoder())
])


#---------------------------
# Full Preprocessing Pipeline
#---------------------------
preprocessing_pipeline = ColumnTransformer(
    transformers=[
        ('num', numeric_pipeline, numeric_features),
        ('cat', categorical_pipeline, categorical_features),
        ('bin', 'passthrough', binary_features)
    ]
)





### Finalize Model Pipeline

In [ ]:
from xgboost import XGBRegressor
from sklearn.compose import TransformedTargetRegressor

xgb_model = XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing_pipeline),
    ('model', TransformedTargetRegressor(
        regressor=xgb_model,
        func=np.log,
        inverse_func=np.exp
    ))
])



In [ ]:
# Train Pipeline
model_pipeline.fit(X_pipeline, y_pipeline)


In [ ]:
# Predicting the Test Set Results

y_pred_pipe = model_pipeline.predict(X_test_pipe)

y_pred_pipe

In [ ]:
# Evaluating pipeline model performance
mae = mean_absolute_error(y_test_pipe, y_pred_pipe)
rmse = np.sqrt(mean_squared_error(y_test_pipe, y_pred_pipe))
r2 = r2_score(y_test_pipe, y_pred_pipe)


print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R² Score: {r2:.2f}")

In [ ]:
# Freezing the Model Pipeline and Feature List for Deployment
import joblib

bundle = {
    "pipeline": model_pipeline,
    "features": X_train.columns.tolist()
}

joblib.dump(bundle, "car_price_xgb_pipeline.pkl")


In [ ]:
prediction = pipeline.predict(input_df)[0]
